# Pipeline de construção


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import requests
from shapely.geometry import shape

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
DATA_DIR = ROOT / 'data'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import COLECOES_RECOMENDADAS, ICECHUNK_PATH, STAC_URL, VARIAVEIS_PRIORITARIAS_POR_COLECAO
from geoparquet import itens_para_geodataframe, salvar_geoparquet
from pipeline import executar_pipeline, validar_cubo
from stac import buscar_itens, conectar_stac
from storage import abrir_repositorio, salvar_dataset_virtual

print('Projeto:', ROOT)
print('Dados:', DATA_DIR)


Projeto: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_v3
Dados: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_v3\data


## Configuração


In [2]:
CODIGO_IBGE_MUNICIPIO = '3549904'
COLECAO = 'S2-16D-2'
VARIAVEIS = VARIAVEIS_PRIORITARIAS_POR_COLECAO[COLECAO]
ICECHUNK_REPO = ROOT / 'data' / 'icechunk_repo'


## Área de estudo


In [ ]:
url_malha = f'https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{CODIGO_IBGE_MUNICIPIO}?formato=application/vnd.geo+json&qualidade=minima'
resposta = requests.get(url_malha, timeout=60)
resposta.raise_for_status()
MUNICIPIO = shape(resposta.json()['features'][0]['geometry'])
print('Área de estudo: São José dos Campos')

Área de estudo: São José dos Campos


## Período


In [ ]:
DATA_FIM = pd.Timestamp.today().normalize()
DATA_INICIO = DATA_FIM - pd.DateOffset(years=10)
print(f'Período: {DATA_INICIO.date()} → {DATA_FIM.date()}')

Período: 2016-09-01 → 2026-09-01


## Consulta STAC


In [ ]:
catalogo = conectar_stac()
cenas = buscar_itens(catalogo, colecao=COLECAO, intersects=MUNICIPIO.__geo_interface__, data_inicio=DATA_INICIO.date().isoformat(), data_fim=DATA_FIM.date().isoformat(), max_itens=None)
print(f'Cenas encontradas: {len(cenas)}')

Cenas encontradas: 221


In [ ]:
if COLECAO not in COLECOES_RECOMENDADAS:
    raise ValueError(f'Coleção não configurada: {COLECAO}')
print('Coleção:', COLECAO)
print('Variáveis solicitadas:', len(VARIAVEIS))

Coleção: S2-16D-2
Variáveis solicitadas: 17


## Variáveis disponíveis


In [7]:
variaveis_configuradas = (
    VARIAVEIS_PRIORITARIAS_POR_COLECAO[COLECAO]
)

variaveis_disponiveis = tuple(
    variavel
    for variavel in variaveis_configuradas
    if any(
        variavel in cena.assets
        for cena in cenas
    )
)

print("Variáveis disponíveis:")

for variavel in variaveis_disponiveis:
    print(" -", variavel)

Variáveis disponíveis:
 - B01
 - B02
 - B03
 - B04
 - B05
 - B06
 - B07
 - B08
 - B8A
 - B09
 - B11
 - B12
 - EVI
 - NBR
 - NDVI
 - SCL
 - CLEAROB


In [8]:
variaveis_disponiveis = tuple(variavel for variavel in VARIAVEIS if any(variavel in cena.assets for cena in cenas))
if not variaveis_disponiveis:
    raise ValueError('Nenhuma variável Sentinel-2 disponível no período selecionado.')
gdf = itens_para_geodataframe(cenas, variaveis=variaveis_disponiveis)
gdf = gdf[gdf.geometry.notna()].copy()
print(f'Registros: {len(gdf)} | Variáveis: {len(variaveis_disponiveis)}')


Registros: 221 | Variáveis: 17


## GeoParquet


In [9]:
caminho_geoparquet = salvar_geoparquet(gdf, ROOT / 'data' / 'geoparquet' / f'{COLECAO}.parquet')
print('GeoParquet salvo:', caminho_geoparquet)


GeoParquet salvo: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_v3\data\geoparquet\S2-16D-2.parquet


## Cubo virtual


In [ ]:
ds = executar_pipeline(collection=COLECAO, assets=variaveis_disponiveis)

PIPELINE SENTINEL-2

Coleção:
  S2-16D-2

Descrição:
  Sentinel-2/MSI — composto de 16 dias

Assets:
  - B01
  - B02
  - B03
  - B04
  - B05
  - B06
  - B07
  - B08
  - B8A
  - B09
  - B11
  - B12
  - EVI
  - NBR
  - NDVI
  - SCL
  - CLEAROB

ETAPA 1 — GEOPARQUET



Registros encontrados: 221

ETAPA 2 — CUBO VIRTUAL
CONSTRUÇÃO DO CUBO MULTIVARIÁVEL

Coleção: S2-16D-2

Assets solicitados:
  - B01
  - B02
  - B03
  - B04
  - B05
  - B06
  - B07
  - B08
  - B8A
  - B09
  - B11
  - B12
  - EVI
  - NBR
  - NDVI
  - SCL
  - CLEAROB

Assets disponíveis:
  ✓ B01
  ✓ B02
  ✓ B03
  ✓ B04
  ✓ B05
  ✓ B06
  ✓ B07
  ✓ B08
  ✓ B8A
  ✓ B09
  ✓ B11
  ✓ B12
  ✓ EVI
  ✓ NBR
  ✓ NDVI
  ✓ SCL
  ✓ CLEAROB

------------------------------------------------------------
Asset: B01
------------------------------------------------------------


c:\Users\giuli\AppData\Local\Programs\Python\Python312\Lib\site-packages\dask\array\chunk_types.py:131: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  import scipy.sparse



------------------------------------------------------------
Asset: B02
------------------------------------------------------------


ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')


...


------------------------------------------------------------
Asset: B09
------------------------------------------------------------


ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')



------------------------------------------------------------
Asset: B11
------------------------------------------------------------


ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')


AsyncTiffException: TimeoutError: 

In [ ]:
validacao = validar_cubo(ds)
print('Dimensões:', validacao['dimensoes'])
print('Variáveis:', validacao['variaveis'])
print('Virtual:', validacao['virtual'])


Dimensões: {'time': 45, 'y': 10560, 'x': 10560}
Variáveis: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'EVI', 'NBR', 'NDVI', 'SCL', 'CLEAROB']
Virtual: {'B01': {'tipo': 'ManifestArray', 'virtual': True}, 'B02': {'tipo': 'ManifestArray', 'virtual': True}, 'B03': {'tipo': 'ManifestArray', 'virtual': True}, 'B04': {'tipo': 'ManifestArray', 'virtual': True}, 'B05': {'tipo': 'ManifestArray', 'virtual': True}, 'B06': {'tipo': 'ManifestArray', 'virtual': True}, 'B07': {'tipo': 'ManifestArray', 'virtual': True}, 'B08': {'tipo': 'ManifestArray', 'virtual': True}, 'B8A': {'tipo': 'ManifestArray', 'virtual': True}, 'B09': {'tipo': 'ManifestArray', 'virtual': True}, 'B11': {'tipo': 'ManifestArray', 'virtual': True}, 'B12': {'tipo': 'ManifestArray', 'virtual': True}, 'EVI': {'tipo': 'ManifestArray', 'virtual': True}, 'NBR': {'tipo': 'ManifestArray', 'virtual': True}, 'NDVI': {'tipo': 'ManifestArray', 'virtual': True}, 'SCL': {'tipo': 'ManifestArray', 'virtua

## Icechunk


In [ ]:
repo = abrir_repositorio(ROOT / "data" / "icechunk_repo")


✓ Repositório aberto: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_v3\data\icechunk_repo


## Persistência


In [ ]:
commit_id = salvar_dataset_virtual(ds, repo,)
print("Commit:", commit_id)


GRAVANDO CUBO VIRTUAL NO ICECHUNK

Variáveis: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'EVI', 'NBR', 'NDVI', 'SCL', 'CLEAROB']
Dimensões: {'time': 45, 'y': 10560, 'x': 10560}
Referências virtuais: 337365
Tamanho lógico: 10,965,000 bytes

Escrevendo referências virtuais...
✓ Referências gravadas.

Realizando commit...

✓ Commit: K27MXNC6Z5TP9WKJG3MG
Commit: K27MXNC6Z5TP9WKJG3MG
